# VQ-VAE 详细说明文档（原理 + 代码 + 训练 + 生成）

这份 notebook 基于项目中的实现：
- `dldemos/VQVAE/model.py`
- `dldemos/VQVAE/main.py`
- `dldemos/VQVAE/configs.py`
- `dldemos/VQVAE/dataset.py`

目标：
1. 讲清楚 VQ-VAE 到底在做什么；
2. 逐段解释项目代码；
3. 给出完整训练与生成流程（两阶段）；
4. 说明常见问题和调参策略。

---

## VQ-VAE 是什么？

VQ-VAE（Vector Quantized Variational AutoEncoder）可以理解为：
- 先用编码器把图像压缩成连续特征；
- 再把连续特征“量化”为离散索引（查码本）；
- 最后用解码器把离散潜变量还原为图像。

与普通 VAE 的关键区别：
- 普通 VAE 的潜变量是连续高斯分布；
- VQ-VAE 的潜变量是**离散符号**（来自 codebook）。

这让 VQ-VAE 非常适合与自回归模型（例如 PixelCNN）结合：
- 在离散潜空间里建模更容易；
- 采样时先生成离散码，再解码为图像。

## 一、VQ-VAE 的数学目标（核心公式）

### 1) 编码与量化
给定输入图像 $x$：
- 编码器输出连续潜变量：$z_e(x)$
- 在码本 $\{e_k\}_{k=1}^{K}$ 中找最近向量：

$$
z_q(x)=e_k,\quad k=\arg\min_j \|z_e(x)-e_j\|_2^2
$$

### 2) 解码重建
解码器输出重建图像：

$$
\hat{x}=\text{Decoder}(z_q(x))
$$

### 3) 损失函数（项目同款）
总损失由三部分组成：

$$
\mathcal{L}=\underbrace{\|x-\hat{x}\|_2^2}_{\text{重建损失}} +
\lambda_e\underbrace{\|\text{sg}[z_e]-z_q\|_2^2}_{\text{embedding损失}} +
\lambda_c\underbrace{\|z_e-\text{sg}[z_q]\|_2^2}_{\text{commitment损失}}
$$

其中 $\text{sg}[\cdot]$ 表示 stop-gradient（停止梯度）。

直观解释：
- 重建损失：要求输出图像接近输入；
- embedding 损失：推动码本向编码器输出靠拢；
- commitment 损失：约束编码器输出不要频繁跳动，稳定贴近选中的码本向量。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# 残差块：用于增强特征表达能力，同时保持训练稳定
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.relu = nn.ReLU()
        # 3x3卷积：局部空间建模
        self.conv1 = nn.Conv2d(dim, dim, 3, 1, 1)
        # 1x1卷积：通道内重组
        self.conv2 = nn.Conv2d(dim, dim, 1)

    def forward(self, x):
        # 残差分支
        tmp = self.relu(x)
        tmp = self.conv1(tmp)
        tmp = self.relu(tmp)
        tmp = self.conv2(tmp)
        # 残差连接
        return x + tmp


# VQ-VAE 主体模型（与项目实现一致，注释更详细）
class VQVAE(nn.Module):
    def __init__(self, input_dim, dim, n_embedding):
        super().__init__()

        # 编码器：两次下采样（stride=2）后接卷积和残差块
        self.encoder = nn.Sequential(
            nn.Conv2d(input_dim, dim, 4, 2, 1),  # H,W -> H/2,W/2
            nn.ReLU(),
            nn.Conv2d(dim, dim, 4, 2, 1),        # H/2,W/2 -> H/4,W/4
            nn.ReLU(),
            nn.Conv2d(dim, dim, 3, 1, 1),
            ResidualBlock(dim),
            ResidualBlock(dim)
        )

        # 码本（离散潜变量词典）：K个向量，每个维度为dim
        self.vq_embedding = nn.Embedding(n_embedding, dim)
        self.vq_embedding.weight.data.uniform_(-1.0 / n_embedding, 1.0 / n_embedding)

        # 解码器：与编码器镜像，使用反卷积逐步上采样
        self.decoder = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1),
            ResidualBlock(dim),
            ResidualBlock(dim),
            nn.ConvTranspose2d(dim, dim, 4, 2, 1),  # H/4,W/4 -> H/2,W/2
            nn.ReLU(),
            nn.ConvTranspose2d(dim, input_dim, 4, 2, 1)  # H/2,W/2 -> H,W
        )

        # 记录下采样次数，用于推断潜空间尺寸
        self.n_downsample = 2

    def forward(self, x):
        # 1) 连续编码
        ze = self.encoder(x)  # [N, C, H, W]

        # 2) 向量量化：为每个空间位置找最近的码本向量
        embedding = self.vq_embedding.weight.data  # [K, C]
        N, C, H, W = ze.shape
        K, _ = embedding.shape

        # 广播后计算距离：distance[n, k, h, w]
        embedding_broadcast = embedding.reshape(1, K, C, 1, 1)
        ze_broadcast = ze.reshape(N, 1, C, H, W)
        distance = torch.sum((embedding_broadcast - ze_broadcast) ** 2, 2)

        # 最近邻索引（离散潜变量）
        nearest_neighbor = torch.argmin(distance, 1)  # [N, H, W]

        # 根据索引查码本，得到量化结果 zq
        zq = self.vq_embedding(nearest_neighbor).permute(0, 3, 1, 2)  # [N, C, H, W]

        # 3) 直通估计器（Straight-Through Estimator）
        # 前向用zq，反向让梯度回到ze
        decoder_input = ze + (zq - ze).detach()

        # 4) 解码重建
        x_hat = self.decoder(decoder_input)
        return x_hat, ze, zq

    @torch.no_grad()
    def encode(self, x):
        # 只输出离散索引（用于后续PixelCNN训练）
        ze = self.encoder(x)
        embedding = self.vq_embedding.weight.data

        N, C, H, W = ze.shape
        K, _ = embedding.shape
        embedding_broadcast = embedding.reshape(1, K, C, 1, 1)
        ze_broadcast = ze.reshape(N, 1, C, H, W)
        distance = torch.sum((embedding_broadcast - ze_broadcast) ** 2, 2)
        nearest_neighbor = torch.argmin(distance, 1)
        return nearest_neighbor  # [N, H, W]

    @torch.no_grad()
    def decode(self, discrete_latent):
        # 从离散索引恢复图像
        zq = self.vq_embedding(discrete_latent).permute(0, 3, 1, 2)
        x_hat = self.decoder(zq)
        return x_hat

    def get_latent_HW(self, input_shape):
        # 输入图像尺寸 -> 离散潜空间尺寸
        C, H, W = input_shape
        return (H // 2 ** self.n_downsample, W // 2 ** self.n_downsample)

## 二、`forward` 的张量流转（以 MNIST 为例）

设输入形状为 `[B, 1, 28, 28]`，`dim=32`：

1. 编码器输出 `ze`：`[B, 32, 7, 7]`
2. 每个 `(h,w)` 位置在码本中找最近向量，得到索引：`[B, 7, 7]`
3. 查码本得到 `zq`：`[B, 32, 7, 7]`
4. 解码器输出重建图 `x_hat`：`[B, 1, 28, 28]`

重点：
- `encode()` 给的是离散索引图（可看作“token map”）；
- `decode()` 负责把 token map 还原成图像；
- 这就是后续“VQ-VAE + PixelCNN”两阶段生成的基础。

## 三、训练阶段1：训练 VQ-VAE（重建器）

这一阶段目标是把 **编码-量化-解码** 学好，让模型能高质量重建输入图像。

项目中的 `train_vqvae` 使用如下损失：
- `l_reconstruct = MSE(x, x_hat)`
- `l_embedding = MSE(ze.detach(), zq)`
- `l_commitment = MSE(ze, zq.detach())`

总损失：

$$
\text{loss}=l_{reconstruct}+\lambda_e l_{embedding}+\lambda_c l_{commitment}
$$

其中项目默认：
- `lambda_e = 1`
- `lambda_c = 0.25`

In [ ]:
import time
import torch
import torch.nn as nn


def train_vqvae(model,
                dataloader,
                device='cuda',
                ckpt_path='dldemos/VQVAE/model_mnist.pth',
                lr=2e-4,
                n_epochs=20,
                l_w_embedding=1.0,
                l_w_commitment=0.25):
    """
    训练 VQ-VAE 的重建器部分（与项目main.py一致）

    参数说明：
    - model: VQVAE模型
    - dataloader: 图像数据加载器（仅图像，不需要类别）
    - device: 训练设备
    - ckpt_path: 权重保存路径
    - lr: 学习率
    - n_epochs: 训练轮数
    - l_w_embedding: embedding loss 权重
    - l_w_commitment: commitment loss 权重
    """
    model = model.to(device)
    model.train()

    optimizer = torch.optim.Adam(model.parameters(), lr)
    mse_loss = nn.MSELoss()

    tic = time.time()
    for epoch in range(n_epochs):
        total_loss = 0.0

        for x in dataloader:
            # 项目里MNIST/CelebA dataloader返回的是图像张量
            x = x.to(device)
            current_batch_size = x.shape[0]

            # 前向：重建图 + 连续潜变量 + 量化潜变量
            x_hat, ze, zq = model(x)

            # 1) 重建损失
            l_reconstruct = mse_loss(x, x_hat)
            # 2) 码本更新项
            l_embedding = mse_loss(ze.detach(), zq)
            # 3) 编码器承诺项
            l_commitment = mse_loss(ze, zq.detach())

            loss = l_reconstruct + l_w_embedding * l_embedding + l_w_commitment * l_commitment

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * current_batch_size

        total_loss /= len(dataloader.dataset)
        torch.save(model.state_dict(), ckpt_path)
        toc = time.time()
        print(f'epoch {epoch} loss: {total_loss:.6f} elapsed {(toc - tic):.2f}s')

    print('VQ-VAE训练完成')

In [ ]:
import cv2
import einops
import numpy as np


@torch.no_grad()
def reconstruct_and_save(model, x, save_path='work_dirs/vqvae_reconstruct_mnist.jpg'):
    """
    可视化重建效果：将原图与重建图拼接保存
    """
    model.eval()
    x_hat, _, _ = model(x)

    # 横向拼接：左边原图，右边重建图
    x_cat = torch.cat((x, x_hat), dim=3)

    # 按网格排布
    n = x.shape[0]
    n1 = int(n ** 0.5)
    x_cat = einops.rearrange(x_cat, '(n1 n2) c h w -> (n1 h) (n2 w) c', n1=n1)

    x_cat = (x_cat.clamp(0, 1) * 255).cpu().numpy().astype(np.uint8)
    cv2.imwrite(save_path, x_cat)
    print(f'重建结果已保存到: {save_path}')

## 四、训练阶段2：在离散潜空间训练 PixelCNN 先验

VQ-VAE 本身解决的是“压缩和重建”，但它并不能直接高质量“无条件生成新图像”。

所以项目采用两阶段：
1. 先训练 VQ-VAE，得到离散潜变量编码；
2. 再训练 `PixelCNNWithEmbedding` 去建模这些离散索引图的分布。

这样采样时就变成：
- PixelCNN 生成离散索引图；
- VQ-VAE `decode()` 把索引图解码为图像。

In [ ]:
from dldemos.VQVAE.pixelcnn_model import PixelCNNWithEmbedding


def train_generative_model(vqvae,
                           gen_model,
                           dataloader,
                           device='cuda',
                           ckpt_path='dldemos/VQVAE/gen_model_mnist.pth',
                           n_epochs=50,
                           lr=1e-3):
    """
    训练离散潜变量上的自回归先验模型（Gated PixelCNN + Embedding）
    """
    vqvae = vqvae.to(device)
    vqvae.eval()  # VQ-VAE 编码器固定

    gen_model = gen_model.to(device)
    gen_model.train()

    optimizer = torch.optim.Adam(gen_model.parameters(), lr)
    loss_fn = nn.CrossEntropyLoss()

    tic = time.time()
    for epoch in range(n_epochs):
        total_loss = 0.0

        for x in dataloader:
            x = x.to(device)
            current_batch_size = x.shape[0]

            # 先把图像编码成离散索引图：shape [B, H_latent, W_latent]
            with torch.no_grad():
                discrete_latent = vqvae.encode(x)

            # PixelCNN 预测每个位置的索引分布
            predict_x = gen_model(discrete_latent)  # [B, K, H_latent, W_latent]

            # 目标就是原始离散索引
            loss = loss_fn(predict_x, discrete_latent)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * current_batch_size

        total_loss /= len(dataloader.dataset)
        torch.save(gen_model.state_dict(), ckpt_path)
        toc = time.time()
        print(f'epoch {epoch} loss: {total_loss:.6f} elapsed {(toc - tic):.2f}s')

    print('先验模型训练完成')

In [ ]:
@torch.no_grad()
def sample_images(vqvae,
                  gen_model,
                  img_shape,
                  n_sample=81,
                  device='cuda',
                  save_path='work_dirs/vqvae_sample_mnist.jpg'):
    """
    先在离散潜空间逐位置采样，再通过VQ-VAE解码成图像
    """
    vqvae = vqvae.to(device)
    vqvae.eval()

    gen_model = gen_model.to(device)
    gen_model.eval()

    # 计算潜空间尺寸
    C, H, W = img_shape
    H_latent, W_latent = vqvae.get_latent_HW((C, H, W))

    # 初始化离散索引图（全0起步）
    x = torch.zeros((n_sample, H_latent, W_latent), dtype=torch.long, device=device)

    # 自回归采样：从左上到右下逐位置采样索引
    for i in range(H_latent):
        for j in range(W_latent):
            output = gen_model(x)  # [B, K, H_latent, W_latent]
            prob_dist = F.softmax(output[:, :, i, j], dim=-1)
            pixel = torch.multinomial(prob_dist, 1)
            x[:, i, j] = pixel[:, 0]

    # 用VQ-VAE解码离散潜变量 -> 图像
    imgs = vqvae.decode(x)

    # 保存为网格图
    imgs = (imgs * 255).clamp(0, 255)
    imgs = einops.rearrange(imgs,
                            '(n1 n2) c h w -> (n1 h) (n2 w) c',
                            n1=int(n_sample ** 0.5))
    imgs = imgs.detach().cpu().numpy().astype(np.uint8)

    cv2.imwrite(save_path, imgs)
    print(f'采样结果已保存到: {save_path}')

## 五、完整使用流程（MNIST示例）

下面这段演示完整 pipeline：
1. 准备数据
2. 初始化 VQ-VAE 与先验模型
3. 训练 VQ-VAE
4. 可视化重建
5. 训练先验 PixelCNN
6. 采样生成新图像

> 注意：如果你只想快速验证流程，可以把 epoch 先设小一些。

In [ ]:
from dldemos.VQVAE.configs import get_cfg
from dldemos.VQVAE.dataset import get_dataloader


# ====== 1) 配置与设备 ======
cfg_id = 0  # 0 对应 MNIST 配置
cfg = get_cfg(cfg_id)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('当前设备:', device)
print('当前配置:', cfg)

# ====== 2) 创建数据加载器 ======
# dataset.py 中返回的是图像tensor，不返回标签
train_loader_stage1 = get_dataloader(cfg['dataset_type'],
                                     cfg['batch_size'],
                                     img_shape=(cfg['img_shape'][1], cfg['img_shape'][2]))
train_loader_stage2 = get_dataloader(cfg['dataset_type'],
                                     cfg['batch_size_2'],
                                     img_shape=(cfg['img_shape'][1], cfg['img_shape'][2]))

# ====== 3) 初始化模型 ======
vqvae = VQVAE(cfg['img_shape'][0], cfg['dim'], cfg['n_embedding'])
gen_model = PixelCNNWithEmbedding(cfg['pixelcnn_n_blocks'],
                                  cfg['pixelcnn_dim'],
                                  cfg['pixelcnn_linear_dim'],
                                  True,
                                  cfg['n_embedding'])

print('VQ-VAE 和先验模型初始化完成')

# ====== 4) 训练阶段1：VQ-VAE ======
# 如果已经训练过，可以注释掉
train_vqvae(vqvae,
            train_loader_stage1,
            device=device,
            ckpt_path=cfg['vqvae_path'],
            lr=cfg['lr'],
            n_epochs=cfg['n_epochs'],
            l_w_embedding=cfg['l_w_embedding'],
            l_w_commitment=cfg['l_w_commitment'])

# ====== 5) 重建可视化 ======
vqvae.load_state_dict(torch.load(cfg['vqvae_path'], map_location=device))
vqvae = vqvae.to(device)

sample_batch = next(iter(train_loader_stage1)).to(device)[:16]
reconstruct_and_save(vqvae, sample_batch, save_path=f"work_dirs/vqvae_reconstruct_{cfg['dataset_type']}.jpg")

# ====== 6) 训练阶段2：离散先验（PixelCNN） ======
train_generative_model(vqvae,
                       gen_model,
                       train_loader_stage2,
                       device=device,
                       ckpt_path=cfg['gen_model_path'],
                       n_epochs=cfg['n_epochs_2'])

# ====== 7) 采样生成 ======
gen_model.load_state_dict(torch.load(cfg['gen_model_path'], map_location=device))
sample_images(vqvae,
              gen_model,
              cfg['img_shape'],
              n_sample=81,
              device=device,
              save_path=f"work_dirs/vqvae_sample_{cfg['dataset_type']}.jpg")

print('完整流程执行完成')

## 六、实战调参建议（非常重要）

### 1) `n_embedding`（码本大小）
- 太小：表达能力不足，重建模糊；
- 太大：训练不稳定、码本利用率可能低；
- 建议：MNIST 可先从 32 或 64 开始。

### 2) `dim`（隐空间通道数）
- 决定编码表达容量；
- 提高 `dim` 通常会提升质量，但显存和计算开销上升。

### 3) `l_w_commitment`
- 太小：编码器输出漂移大；
- 太大：编码器被约束过强，可能损失重建能力；
- 项目默认 0.25 是常见稳定值。

### 4) 两阶段训练时长
- 第一阶段（VQ-VAE）不收敛，第二阶段再努力也没用；
- 先确保重建效果够好，再训练 PixelCNN 先验。

### 5) 快速验证流程建议
- 先把 `n_epochs` 和 `n_epochs_2` 改小（例如 1~3）做 smoke test；
- 流程跑通后再开长训。

## 七、常见问题排查（FAQ）

### Q1. 为什么重建图像很糊？
- 先检查第一阶段是否训练充分；
- 查看 `dim` 是否太小；
- 检查码本大小和损失权重是否合理。

### Q2. 生成图像质量差，但重建还行？
- 这通常是第二阶段先验模型没学好；
- 增加 `n_epochs_2`，或提升 PixelCNN 容量（块数/通道）。

### Q3. 出现码本塌缩（只用少数码字）怎么办？
- 适当增大学习率 warmup 或调整 `l_w_commitment`；
- 提升 batch size 有时也会改善码本利用。

### Q4. 为什么要先离散化再建模？
- 直接在像素空间自回归开销大；
- 在低分辨率离散潜空间建模，效率更高、语义更强。

### Q5. 这个项目能做条件生成吗？
- 可以，在先验模型中加入类别条件（embedding）即可；
- 思路与 conditional PixelCNN 类似。

## 八、总结

你可以把这套方法记成一句话：

**VQ-VAE 负责“把图像变成离散token并可逆还原”，PixelCNN 负责“在token空间生成新样本”。**

这正是很多现代生成模型（离散token路线）的核心思想雏形。

如果你愿意，下一步可以继续做：
1. 把这份 notebook 扩展成“条件 VQ-VAE + 条件先验”版本；
2. 加入码本利用率统计（perplexity）监控；
3. 迁移到 CelebAHQ 配置并比较不同 `n_embedding` 效果。